# Notebook 02: Transformer Architecture

## Why This Matters
Every modern LLM is a transformer. Knowing the architecture at the code level — not just the diagram level — means you can reason about memory usage, compute costs, why attention is O(N²), what the residual stream is actually doing, and where inference optimizations plug in. This notebook builds a GPT-style decoder-only transformer from scratch, then maps each component to the production code you'll find in HuggingFace and vLLM.

## Structure
1. Token embeddings and positional encodings
2. Scaled dot-product attention — from math to code
3. Multi-head attention
4. Feed-forward (MLP) block
5. LayerNorm and Pre-LN vs Post-LN
6. Residual stream — the central abstraction
7. Full GPT-style decoder block
8. Causal masking and autoregressive generation
9. Parameter count math

## What You'll Learn
- Why attention is O(N²) in memory and how that motivates Flash Attention (notebook 06)
- What the residual stream is and why it's the right mental model for LLM internals
- How Pre-LN stabilizes training vs. the original Post-LN
- How to count parameters for any transformer config
- The exact shape of every tensor flowing through the model

## Background

### What is a transformer?

A transformer is a neural network architecture that processes sequences by letting every element attend to every other element simultaneously — rather than reading left-to-right one step at a time, like the RNNs it replaced. The output of each layer is a weighted mixture of all input positions, where the weights are computed dynamically from the content of the sequence itself.

The transformer was introduced in the 2017 paper *"Attention Is All You Need"* (Vaswani et al., Google Brain). At the time, the dominant approach to sequence modeling was LSTMs and GRUs — gated recurrent networks that processed tokens sequentially. The transformer's bet was that sequential processing was the bottleneck, and that replacing it with fully parallel attention would be both faster and better.

That bet paid off comprehensively. By 2019, BERT and GPT-2 had demonstrated that transformers trained on raw text at scale could solve nearly any NLP task. Every major language model since — GPT-3, GPT-4, LLaMA, Claude, Gemini — is a transformer.

### Why did attention beat RNNs?

RNNs had a **vanishing gradient problem**: information about tokens early in a sequence had to travel through many recurrent steps to influence the output, and gradients (the training signal) diminished exponentially along the way. LSTMs and GRUs partially solved this with gating mechanisms, but long-range dependencies remained difficult.

Attention solved it directly: every token has a **direct path** to every other token in a single layer. The word "bank" in "I went to the bank to deposit money" can directly attend to "deposit" and "money" to resolve its meaning, with no intervening steps.

RNNs also couldn't be parallelized across sequence positions — each step depended on the previous. Transformers compute all positions simultaneously, which maps well to GPU hardware built for massive parallel computation.

### The decoder-only architecture

The original transformer was encoder-decoder: an encoder read the input sequence, and a decoder generated the output. This works well for translation (fixed input → fixed output). But for open-ended text generation, the encoder-decoder split is unnecessary overhead.

GPT (2018) introduced the **decoder-only** transformer: a single stack of layers that processes the prompt and generates new tokens left-to-right. There's no separate encoder — the same weights handle both understanding and generation. All modern LLMs (GPT, LLaMA, Mistral, Claude, Gemini) use decoder-only architectures.

BERT used encoder-only — good for classification and understanding tasks, but can't generate text.

### The residual stream mental model

The most useful mental model for understanding transformer internals (and for debugging them) is the **residual stream**: a vector of size `d_model` that flows through the entire network, and which each layer reads from and writes back to. Attention heads and MLP blocks don't transform the stream — they compute a *delta* and add it. The final output is the sum of all these deltas across all layers.

This framing, introduced by Elhage et al. (Anthropic, 2021) in *"A Mathematical Framework for Transformer Circuits"*, makes it much easier to reason about what information is preserved, what gets modified, and why certain heads do what they do.

In [ ]:
%pip install -q torch transformers

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

# Use MPS on Apple Silicon, CUDA if available, else CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## 1. Token Embeddings and Positional Encodings

The first operation in every transformer: convert integer token IDs into continuous vectors.

```
tokens:     [1024,  531, 8921,   42]      shape: (batch, seq_len)
                ↓  nn.Embedding
embeddings: [[...], [...], [...], [...]]  shape: (batch, seq_len, d_model)
                ↓  + positional encoding
x:          [[...], [...], [...], [...]]  shape: (batch, seq_len, d_model)
```

Two positional encoding strategies:
- **Learned absolute** (GPT-2, original BERT): a separate embedding table for positions 0..max_seq_len
- **RoPE** (LLaMA, Mistral, GPT-NeoX): rotary embeddings baked into attention — covered in notebook 16

GPT-2 style (simplest, good for learning the architecture):

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int = 50_257     # GPT-2 vocabulary
    max_seq_len: int = 1024      # maximum context length
    d_model: int = 768           # embedding / hidden dimension
    n_heads: int = 12            # number of attention heads
    n_layers: int = 12           # number of transformer blocks
    d_ff: int = 3072             # feed-forward inner dimension (4 × d_model)
    dropout: float = 0.1

    @property
    def d_head(self):
        assert self.d_model % self.n_heads == 0
        return self.d_model // self.n_heads


cfg = GPTConfig()
print(f"GPT-2 config:")
print(f"  d_model={cfg.d_model}, n_heads={cfg.n_heads}, d_head={cfg.d_head}")
print(f"  n_layers={cfg.n_layers}, d_ff={cfg.d_ff}")
print(f"  vocab_size={cfg.vocab_size}, max_seq_len={cfg.max_seq_len}")

In [ ]:
class TokenEmbedding(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        # idx: (batch, seq_len) integer token IDs
        B, T = idx.shape
        positions = torch.arange(T, device=idx.device)  # (T,)
        tok = self.tok_emb(idx)        # (B, T, d_model)
        pos = self.pos_emb(positions)  # (T, d_model) — broadcasts over batch
        return self.drop(tok + pos)    # (B, T, d_model)


# Smoke test
emb = TokenEmbedding(cfg)
dummy_tokens = torch.randint(0, cfg.vocab_size, (2, 16))  # batch=2, seq_len=16
out = emb(dummy_tokens)
print(f"Input shape:  {dummy_tokens.shape}  (batch=2, seq_len=16)")
print(f"Output shape: {out.shape}  (batch=2, seq_len=16, d_model=768)")

## 2. Scaled Dot-Product Attention

The core operation in every transformer layer:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In plain English:
1. For each query position, compute a dot product against every key position → raw similarity scores
2. Scale by $\sqrt{d_k}$ to prevent softmax saturation at high dimensions
3. Apply softmax → attention weights (sum to 1 per query position)
4. Weighted sum of values → output

```
Q: (batch, heads, seq, d_head)   ← "what am I looking for?"
K: (batch, heads, seq, d_head)   ← "what do I contain?"
V: (batch, heads, seq, d_head)   ← "what do I communicate?"

QK^T: (batch, heads, seq, seq)   ← O(N²) — this is the memory bottleneck
```

The `seq × seq` attention matrix is why vanilla attention scales quadratically. Flash Attention (notebook 06) avoids materializing this full matrix.

In [ ]:
def scaled_dot_product_attention(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    mask: torch.Tensor = None,
    dropout: float = 0.0,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Args:
        q, k, v: (batch, heads, seq, d_head)
        mask: (1, 1, seq, seq) boolean, True = keep
    Returns:
        output: (batch, heads, seq, d_head)
        attn_weights: (batch, heads, seq, seq)
    """
    d_k = q.shape[-1]
    scale = math.sqrt(d_k)

    # (batch, heads, seq_q, seq_k)
    scores = torch.matmul(q, k.transpose(-2, -1)) / scale

    if mask is not None:
        scores = scores.masked_fill(~mask, float('-inf'))

    attn_weights = F.softmax(scores, dim=-1)

    if dropout > 0.0 and torch.is_grad_enabled():
        attn_weights = F.dropout(attn_weights, p=dropout)

    output = torch.matmul(attn_weights, v)  # (batch, heads, seq, d_head)
    return output, attn_weights


# Shape walkthrough with concrete numbers
batch, heads, seq, d_head = 2, 12, 16, 64
q = torch.randn(batch, heads, seq, d_head)
k = torch.randn(batch, heads, seq, d_head)
v = torch.randn(batch, heads, seq, d_head)

out, weights = scaled_dot_product_attention(q, k, v)
print(f"Q/K/V shape:        {q.shape}")
print(f"Attention weights:  {weights.shape}  ← the O(N²) matrix")
print(f"Output shape:       {out.shape}")
print(f"\nMemory for attn matrix (float32): {weights.numel() * 4 / 1024:.1f} KB")
print(f"Memory for attn matrix at seq=4096: {batch * heads * 4096 * 4096 * 4 / 1e9:.2f} GB  ← why Flash Attention matters")

## 3. Causal Masking

Autoregressive (decoder-only) models can only attend to past positions — each token predicts the next one, so attending to future tokens would be cheating.

```
Causal mask for seq_len=4:

          pos 0  pos 1  pos 2  pos 3
pos 0  [  True  False  False  False ]
pos 1  [  True   True  False  False ]
pos 2  [  True   True   True  False ]
pos 3  [  True   True   True   True ]

Positions marked False become -inf before softmax → weight ≈ 0 after softmax
```

In [ ]:
def make_causal_mask(seq_len: int, device: torch.device) -> torch.Tensor:
    """Lower-triangular boolean mask. True = attend, False = mask out."""
    return torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool, device=device))


mask = make_causal_mask(6, device=torch.device("cpu"))
print("Causal mask (6×6):")
print(mask.int())

# Apply to attention scores and verify future tokens get ~0 weight
scores = torch.zeros(1, 1, 6, 6)  # flat scores
masked_scores = scores.masked_fill(~mask, float('-inf'))
weights = F.softmax(masked_scores, dim=-1)
print(f"\nAttention weights (uniform scores + causal mask):")
print(weights.squeeze().round(decimals=3))

## 4. Multi-Head Attention

Instead of one attention operation with full `d_model` dimensions, we run `n_heads` parallel attention operations each with `d_head = d_model / n_heads` dimensions. This lets the model attend to multiple positions for different reasons simultaneously.

```
x: (B, T, d_model)
  → linear projections → Q, K, V: (B, T, d_model)
  → reshape → (B, n_heads, T, d_head)
  → scaled dot-product attention per head
  → concat heads → (B, T, d_model)
  → output projection → (B, T, d_model)
```

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.n_heads = cfg.n_heads
        self.d_head = cfg.d_head
        self.d_model = cfg.d_model

        # Fused QKV projection — one matrix multiply instead of three
        self.qkv_proj = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.dropout = cfg.dropout

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        B, T, C = x.shape  # C = d_model

        # Project to Q, K, V in one shot, then split
        qkv = self.qkv_proj(x)                        # (B, T, 3*d_model)
        q, k, v = qkv.split(self.d_model, dim=-1)     # each: (B, T, d_model)

        # Reshape for multi-head: (B, T, d_model) → (B, n_heads, T, d_head)
        def reshape_for_heads(t):
            return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        q, k, v = reshape_for_heads(q), reshape_for_heads(k), reshape_for_heads(v)

        # Attend
        attn_out, _ = scaled_dot_product_attention(q, k, v, mask=mask, dropout=self.dropout)

        # Merge heads: (B, n_heads, T, d_head) → (B, T, d_model)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, C)

        return self.out_proj(attn_out)  # (B, T, d_model)


mha = MultiHeadAttention(cfg)
x = torch.randn(2, 16, cfg.d_model)  # batch=2, seq=16
mask = make_causal_mask(16, device=torch.device("cpu")).unsqueeze(0).unsqueeze(0)
out = mha(x, mask)
print(f"MHA input:  {x.shape}")
print(f"MHA output: {out.shape}")
print(f"MHA params: {sum(p.numel() for p in mha.parameters()):,}")

## 5. Feed-Forward (MLP) Block

Each transformer layer has an MLP block after attention. In GPT-2 style:

```
x → Linear(d_model → d_ff) → GELU → Linear(d_ff → d_model)
```

`d_ff` is typically `4 × d_model`. This is where most of the model's "knowledge storage" happens — the MLP blocks act as associative memories (Geva et al., 2021).

Modern models (LLaMA, Mistral) use **SwiGLU** instead of GELU:
```
x → [Linear(d_model → d_ff) → SiLU] × Linear(d_model → d_ff) → Linear(d_ff → d_model)
```
SwiGLU adds a gating mechanism — one branch gates the other — and empirically improves quality.

In [ ]:
class MLP(nn.Module):
    """GPT-2 style MLP with GELU activation."""
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.fc1  = nn.Linear(cfg.d_model, cfg.d_ff)
        self.fc2  = nn.Linear(cfg.d_ff, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)              # (B, T, d_ff)
        x = F.gelu(x)                # non-linearity
        x = self.drop(x)
        x = self.fc2(x)              # (B, T, d_model)
        return x


class SwiGLU(nn.Module):
    """LLaMA / Mistral style MLP with SwiGLU gating."""
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        # Two parallel up-projections: one is gated by SiLU of the other
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj   = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate = F.silu(self.gate_proj(x))  # (B, T, d_ff)
        up   = self.up_proj(x)            # (B, T, d_ff)
        return self.down_proj(gate * up)  # (B, T, d_model)


mlp_gpt  = MLP(cfg)
mlp_swi  = SwiGLU(cfg.d_model, cfg.d_ff)
x = torch.randn(2, 16, cfg.d_model)

print(f"GPT-2 MLP params: {sum(p.numel() for p in mlp_gpt.parameters()):,}")
print(f"SwiGLU MLP params: {sum(p.numel() for p in mlp_swi.parameters()):,}")
print(f"\nMLP output shape: {mlp_gpt(x).shape}")

## 6. LayerNorm: Pre-LN vs. Post-LN

The original transformer (Vaswani 2017) used **Post-LN** — LayerNorm after the residual addition:
```
Post-LN:  x = LayerNorm(x + sublayer(x))
```

Modern LLMs use **Pre-LN** — LayerNorm before the sublayer:
```
Pre-LN:   x = x + sublayer(LayerNorm(x))
```

Pre-LN is more stable to train (gradients flow more easily through the residual connection) and allows training without warmup. GPT-2, GPT-3, LLaMA, and virtually all modern LLMs use Pre-LN.

LLaMA also uses **RMSNorm** instead of LayerNorm — removes the mean-centering step, slightly faster:
```
RMSNorm(x) = x / RMS(x) * γ   where RMS(x) = sqrt(mean(x²))
```

In [ ]:
class RMSNorm(nn.Module):
    """Root Mean Square LayerNorm — used in LLaMA/Mistral."""
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return self.weight * (x / rms)


# Compare LayerNorm vs RMSNorm
x = torch.randn(2, 16, cfg.d_model)
ln = nn.LayerNorm(cfg.d_model)
rn = RMSNorm(cfg.d_model)

print(f"LayerNorm params: {sum(p.numel() for p in ln.parameters()):,}  (weight + bias)")
print(f"RMSNorm params:   {sum(p.numel() for p in rn.parameters()):,}  (weight only)")
print(f"Output shape: {ln(x).shape}")

## 7. The Residual Stream

The residual stream is the central abstraction for understanding transformer internals (Elhage et al., "A Mathematical Framework for Transformer Circuits", 2021).

```
x₀ = embed(tokens)                      ← initial residual stream
x₁ = x₀ + attn₁(LN(x₀))               ← layer 1 attention writes to stream
x₁ = x₁ + mlp₁(LN(x₁))                ← layer 1 MLP writes to stream
x₂ = x₁ + attn₂(LN(x₁))               ← layer 2 attention reads x₁, writes delta
...
xₙ = xₙ₋₁ + mlpₙ(LN(xₙ₋₁))
logits = unembed(LN(xₙ))               ← project back to vocabulary
```

Key insight: each attention head and MLP reads from the stream and adds a **residual delta** — never replaces, always adds. The final output is the sum of all these deltas.

## 8. Full GPT-Style Decoder Block

In [ ]:
class TransformerBlock(nn.Module):
    """GPT-2 style Pre-LN transformer block."""
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.ln1  = nn.LayerNorm(cfg.d_model)
        self.attn = MultiHeadAttention(cfg)
        self.ln2  = nn.LayerNorm(cfg.d_model)
        self.mlp  = MLP(cfg)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        # Pre-LN: normalize BEFORE sublayer, add residual AFTER
        x = x + self.attn(self.ln1(x), mask)  # attention residual
        x = x + self.mlp(self.ln2(x))          # MLP residual
        return x


class GPT(nn.Module):
    """Minimal GPT-style decoder-only transformer."""
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg
        self.embed = TokenEmbedding(cfg)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f  = nn.LayerNorm(cfg.d_model)   # final layer norm
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        # Weight tying: input embedding and output projection share weights
        # This is standard practice — halves the parameter count for the vocab projection
        self.lm_head.weight = self.embed.tok_emb.weight

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        mask = make_causal_mask(T, idx.device).unsqueeze(0).unsqueeze(0)

        x = self.embed(idx)              # (B, T, d_model)
        for block in self.blocks:
            x = block(x, mask)           # (B, T, d_model)
        x = self.ln_f(x)                 # (B, T, d_model)
        logits = self.lm_head(x)         # (B, T, vocab_size)
        return logits

    @torch.no_grad()
    def generate(self, idx: torch.Tensor, max_new_tokens: int, temperature: float = 1.0) -> torch.Tensor:
        """Greedy/temperature sampling — autoregressive generation."""
        for _ in range(max_new_tokens):
            # Crop to max context length
            idx_cond = idx[:, -self.cfg.max_seq_len:]
            logits = self(idx_cond)                   # (B, T, vocab_size)
            logits = logits[:, -1, :] / temperature   # last position only
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat([idx, next_token], dim=1)
        return idx


model = GPT(cfg)
print(f"Model parameter count: {sum(p.numel() for p in model.parameters()):,}")

## 9. Parameter Count Math

Understanding where parameters live is essential for memory budgeting and inference optimization.

For a transformer with `n_layers` layers, `d_model` hidden dim, `d_ff = 4 × d_model`, `n_heads` heads:

```
Per layer:
  Attention QKV proj:  d_model × 3 × d_model  =  3 × d_model²
  Attention out proj:  d_model × d_model       =  d_model²
  MLP fc1:             d_model × d_ff          =  4 × d_model²
  MLP fc2:             d_ff × d_model          =  4 × d_model²
  LayerNorms (×2):     2 × 2 × d_model         ≈  0 (negligible)
  Total per layer:     ≈ 12 × d_model²

Non-layer:
  Token embedding:     vocab_size × d_model
  Position embedding:  max_seq_len × d_model
  LM head:             (tied to token embedding)
```

In [ ]:
def count_params_breakdown(model: GPT) -> None:
    """Print a parameter breakdown by component."""
    embed_params = sum(p.numel() for p in model.embed.parameters())
    block_params = sum(p.numel() for p in model.blocks.parameters())
    head_params  = sum(p.numel() for p in model.lm_head.parameters())
    total        = sum(p.numel() for p in model.parameters())

    # lm_head is weight-tied so doesn't add to total
    unique_total = embed_params + block_params + sum(p.numel() for p in model.ln_f.parameters())

    print(f"Parameter breakdown:")
    print(f"  Embeddings (tok + pos): {embed_params:>12,}  ({100*embed_params/unique_total:.1f}%)")
    print(f"  Transformer blocks:     {block_params:>12,}  ({100*block_params/unique_total:.1f}%)")
    print(f"  Final LayerNorm:        {sum(p.numel() for p in model.ln_f.parameters()):>12,}")
    print(f"  LM head (tied, no add): {head_params:>12,}")
    print(f"  {'─'*40}")
    print(f"  Unique total:           {unique_total:>12,}")
    print(f"  At BF16 (2 bytes):      {unique_total * 2 / 1e9:>11.2f} GB")
    print()

    # Per-layer breakdown
    layer = model.blocks[0]
    attn_p = sum(p.numel() for p in layer.attn.parameters())
    mlp_p  = sum(p.numel() for p in layer.mlp.parameters())
    ln_p   = sum(p.numel() for p in layer.ln1.parameters()) + sum(p.numel() for p in layer.ln2.parameters())
    print(f"Per-layer breakdown (layer 0):")
    print(f"  Attention:   {attn_p:>10,}  ({100*attn_p/(attn_p+mlp_p+ln_p):.1f}%)")
    print(f"  MLP:         {mlp_p:>10,}  ({100*mlp_p/(attn_p+mlp_p+ln_p):.1f}%)")
    print(f"  LayerNorms:  {ln_p:>10,}")


count_params_breakdown(model)

# Verify against theoretical formula: ≈ 12 × n_layers × d_model² + vocab × d_model
theoretical = 12 * cfg.n_layers * cfg.d_model**2 + cfg.vocab_size * cfg.d_model
print(f"\nTheoretical estimate (12 × L × d²): {theoretical:,}")

In [ ]:
# Verify forward pass and generation work
model.eval()

# Forward pass
tokens = torch.randint(0, cfg.vocab_size, (1, 8))
with torch.no_grad():
    logits = model(tokens)
print(f"Forward pass:")
print(f"  Input:   {tokens.shape}")
print(f"  Logits:  {logits.shape}  (batch=1, seq=8, vocab={cfg.vocab_size})")
print(f"  Logits sum to softmax-normalized: {F.softmax(logits[0, -1], dim=-1).sum().item():.4f}")

# Autoregressive generation
prompt = torch.randint(0, cfg.vocab_size, (1, 4))
generated = model.generate(prompt, max_new_tokens=8, temperature=0.8)
print(f"\nGeneration:")
print(f"  Prompt:    {prompt.shape} → {prompt[0].tolist()}")
print(f"  Generated: {generated.shape} → {generated[0].tolist()}")
print(f"  New tokens: {generated[0, 4:].tolist()}")

## Summary

```
Token IDs
    ↓  tok_emb + pos_emb
Residual stream x₀: (B, T, d_model)
    ↓  × n_layers:
    |   x = x + Attn(LN(x))   ← attention reads and writes to stream
    |   x = x + MLP(LN(x))    ← MLP reads and writes to stream
    ↓
Final LN
    ↓
LM head (linear)
    ↓
Logits: (B, T, vocab_size)
```

**Key numbers to remember:**
- ~12 × L × d² parameters per model (non-embedding)
- O(N²) memory for the attention matrix — the Flash Attention target
- Pre-LN (modern) vs Post-LN (original) — virtually all production models use Pre-LN
- Weight tying: input embedding = output projection (transposed)

**Next:** Notebook 03 covers sampling — how you convert the logit distribution into actual tokens at inference time (temperature, top-k, top-p, and their tradeoffs).